In [90]:
import os
from openai  import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [91]:
import chromadb

client = chromadb.CloudClient(
  api_key='ck-Di9AnEsphUuwxi2Fdz2bS5fowtQWwYP9w4anm5MFYEX7',
  tenant='bbe073fa-eced-4068-86e9-e22cb9a165e8',
  database='Meeting'
)

In [9]:
print(client.heartbeat)

<bound method Client.heartbeat of <chromadb.api.client.Client object at 0x00000254CCED0DA0>>


In [11]:
collection = client.create_collection(name="my_collection")

In [92]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

collection = client.create_collection(
    name="my_collection",
    embedding_function=OpenAIEmbeddingFunction(
        model_name="text-embedding-3-small"
    )
)

In [69]:
openai = OpenAI()

In [72]:
from groq import Groq

client = Groq()
data = ""
with open("videoplayback.mp4", "rb") as file:
    transcription = openai.audio.transcriptions.create(
      model="gpt-4o-transcribe",
      file=file,
    )
    data += (transcription.text)
      

In [83]:
rawdata = data

In [84]:
rawdata

['स्वागत',
 'है',
 'आप',
 'सबी',
 'का',
 'चाय',
 'और',
 'कोड',
 'में',
 'और',
 'अगर',
 'आप',
 'फॉलो',
 'कर',
 'रहे',
 'हैं',
 'चैनल',
 'को',
 'तो',
 'मैंने',
 'बैंगलूरू',
 'में',
 'एक',
 'सीरीस',
 'स्टार्ट',
 'करी',
 'थी',
 'uncomfortable',
 'javascript',
 'जहांपे',
 'मैं',
 'आपको',
 'javascript',
 'डिफरंट',
 'तरीके',
 'से',
 'लिखना',
 'सिखा',
 'रहा',
 'था',
 'जिनको',
 'patterns',
 'भी',
 'बोला',
 'जाता',
 'है',
 'usually',
 'जावा',
 'वाले',
 'लोग',
 'काफ़ी',
 'इन',
 'patterns',
 'को',
 'पढ़ते',
 'हैं',
 'क्योंकि',
 'वहाँ',
 'पर',
 'use',
 'cases',
 'बहुत',
 'जादा',
 'आते',
 'हैं',
 'इनके',
 'लेकिन',
 'javascript',
 'वाले',
 'usually',
 'लोग',
 'initially',
 'तो',
 'नहीं',
 'लिखते',
 'हैं',
 'जब',
 'आप',
 'किसी',
 'library',
 'को',
 'maintain',
 'करने',
 'लग',
 'जाते',
 'हैं',
 'framework',
 'में',
 'जाते',
 'हैं',
 'या',
 'फिर',
 'बड़ा',
 'codebase',
 'देखते',
 'हैं',
 'तब',
 'आपको',
 'इन',
 'patterns',
 'introduction',
 'मिलता',
 'है',
 'पर',
 'क्योंकि',
 'अब',
 'हम',
 'आ',
 'गए',
 '

In [86]:


data = " ".join(rawdata)  

chunk_size = 300      
overlap = 50

chunks = []
ids = []

i = 0
chunk_index = 0

while i < len(data):
    text = data[i:i + chunk_size]

    chunks.append(text)
    ids.append(f"id_{chunk_index}")

    chunk_index += 1
    i += (chunk_size - overlap)

print("Total chunks:", len(chunks))
print("Sample chunk:", chunks[0])

response = openai.embeddings.create(
    input=chunks,
    model="text-embedding-3-small"
)

embeddings = [item.embedding for item in response.data]

print(len(ids), len(chunks), len(embeddings))

Total chunks: 34
Sample chunk: स्वागत है आप सबी का चाय और कोड में और अगर आप फॉलो कर रहे हैं चैनल को तो मैंने बैंगलूरू में एक सीरीस स्टार्ट करी थी uncomfortable javascript जहांपे मैं आपको javascript डिफरंट तरीके से लिखना सिखा रहा था जिनको patterns भी बोला जाता है usually जावा वाले लोग काफ़ी इन patterns को पढ़ते हैं क्योंकि वहाँ पर
34 34 34


In [93]:
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=chunks,
)

In [42]:
embeddings = response.data[0].embedding

In [54]:
print(len(embeddings[0]))

1536


In [113]:
system_prompt = f"""If the user input is a greeting (like hello, hi, thanks), respond normally like a human and DO NOT use context.

Only use the provided context when the user asks about concepts or content from the video.
"""



In [103]:
def get_context(message):
    relevant_data = collection.query(
    query_texts=[message]
    )
    result = " ".join(
    " ".join(inner) for inner in relevant_data.get("documents")) 
    return result

In [120]:
get_context("log in patterns को पढ़ते हैं क्योंकि वहाँ पर use cases बहुत जादा")

'लोग काफ़ी इन patterns को पढ़ते हैं क्योंकि वहाँ पर use cases बहुत जादा आते हैं इनके लेकिन javascript वाले usually लोग initially तो नहीं लिखते हैं जब आप किसी library को maintain करने लग जाते हैं framework में जाते हैं या फिर बड़ा codebase देखते हैं तब आपको इन patterns introduction मिलता है पर क्योंकि  इतना difficult नहीं था बट यह actually मैं patterns की study होती है बखायदा books हैं इनके उपर कि पूरा पूरा काम ऐसे pattern adapter से होता है अब यहां तक तो मामला ठीक है आपको भी basic लगा होगा लेकिन अब हम क्या करते हैं एक practical use case और लेते हैं यह भी practical है 100% practical है लेकिन कैसा है कैसे लिखा जाता है सब को समझ में आएगा आपको हम एक practical use case भी लेंगे इसके लिए बड़ा ही मज़ाएगा आपको तो पहले मैं क्या करता हूँ सबसे पहले मैं आपको इसका एक basic example देता हूँ कि ये adapter pattern है क्या था कि atleast हमें click करना शुरू उसके बाद एक real world cases लेंगे खर मैं नए syste  ही interesting pattern है almost हर professional code base में ये pattern use किया जाता है लिखा

In [114]:
from groq import Groq

groq = Groq()


def chat(message):
    context = get_context(message);
    response = groq.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
        {
            "role": "system",
            "content": system_prompt + "\n" + context
        },{"role":"user","content":message}
        ],
    )
    return response.choices[0].message.content

In [119]:
chat("log in patterns को पढ़ते हैं क्योंकि वहाँ पर use cases बहुत जादा")

'बिल्कुल, चलिए\u202f**Adapter Pattern**\u202fको अच्छे से समझते हैं और फिर एक छोटा‑छोटा प्रैक्टिकल उदाहरण देखते हैं, जैसे‑कि आप ने “weather API” और “local storage simulator” का जिक्र किया था।\n\n---\n\n## 1. Adapter Pattern – क्या है?\n\n**Adapter**\u202fएक डिज़ाइन‑पैटर्न है जिसका काम दो *incompatible*\u202fइंटरफ़ेस (या API) को एक‑दूसरे के साथ काम करने लायक बनाना है।  \nइसे अक्सर **Wrapper**\u202fभी कहा जाता है – आप एक “अडैप्टर” बनाते हैं जो मौजूदा क्लास/फ़ंक्शन की API को आपके एप्लिकेशन की ज़रूरतों के अनुसार “अडैप्ट” (अनुकूल) करता है।\n\n### क्यों जरूरत पड़ती है?\n\n| परिदृश्य | बिना Adapter | Adapter के साथ |\n|----------|-------------|----------------|\n| अलग‑अलग लाइब्रेरी/सर्विसेज़ की अलग‑अलग मेथड सिग्नेचर | कोड में हर बार कंडीशनल चेक, दोहराव | एक ही यूनिफॉर्म इंटरफ़ेस → कोड साफ़, टेस्टिंग आसान |\n| थर्ड‑पार्टी API बदलता है | बहुत सारा कोड ब्रेक हो जाता है | केवल Adapter को बदलें, बाकी कोड रहता है वैसा ही |\n| मोबाइल/डेस्कटॉप में एक ही बिज़नेस लॉजिक को दो अलग प्लेटफ़ॉर्म पर चलाना | द